# 🏠 Automated Mortgage Loan System (Google Colab)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cosmos117/Automated-Mortgage-Loan-System/blob/feature/full-pipeline/Mortgage_Loan_System_Colab.ipynb)

This notebook runs the whole system in Colab:

1. **Ingest** loan applications and the lending policy (OCR for scanned forms)
2. **Check** an application against ~25 lending rules → `COMPLIANT` / `NEEDS MANUAL REVIEW` / `NOT COMPLIANT`
3. **Ask** questions in plain English (RAG with citations)
4. **Read** a scanned Form 1003 into structured data
5. **Measure** answer accuracy

**Before you start:** `Runtime → Change runtime type → T4 GPU` (faster answers).
Then `Runtime → Run all`. The first run downloads ~3 GB of models and OCRs the sample scan (about 5 minutes in total).

## 1. Get the code

In [ ]:
import os, sys

IN_COLAB = "google.colab" in sys.modules

REPO_URL = "https://github.com/cosmos117/Automated-Mortgage-Loan-System.git"
BRANCH = "feature/full-pipeline"          # change to "main" once the branch is merged
PROJECT_DIR = "/content/Automated-Mortgage-Loan-System"

if IN_COLAB:
    if not os.path.exists(PROJECT_DIR):
        !git clone -q --branch {BRANCH} {REPO_URL} {PROJECT_DIR}
    %cd {PROJECT_DIR}
else:
    print("Not in Colab: using the project in", os.getcwd())

> **Repository private, or branch not pushed?** Zip the project folder on your computer
> (without `.venv` and `output`), upload it with the 📁 Files panel on the left, then run
> `!unzip -q /content/Automated-Mortgage-Loan-System.zip -d /content` and re-run the cell above.

## 2. Install

In [ ]:
if IN_COLAB:
    !apt-get -qq install -y tesseract-ocr > /dev/null      # OCR program for scanned forms
    !pip install -q -r requirements.txt

import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (CPU: slower but works)")

In [ ]:
# Quick self-test (28 tests, no models needed, ~10 s)
import subprocess
result = subprocess.run([sys.executable, "-m", "unittest", "discover", "tests"], capture_output=True, text=True)
print("\n".join(result.stderr.strip().splitlines()[-3:]))

## 3. (Optional) Keep results on Google Drive

Colab deletes local files when the runtime resets. Set `SAVE_TO_DRIVE = True` to keep the
database, OCR cache and reports in your Drive. **Run this before step 4.**

In [ ]:
SAVE_TO_DRIVE = False

if IN_COLAB and SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    os.environ["MORTGAGE_OUTPUT_DIR"] = "/content/drive/MyDrive/mortgage_loan_output"

os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
os.environ["TRANSFORMERS_VERBOSITY"] = "error"

## 4. Load the documents

Reads every application in `data/applications/` and the policy in `Regulations/`,
splits them into chunks and stores them in the vector database.
The 9-page scan takes about a minute to OCR the first time; unchanged files are skipped afterwards.

In [ ]:
from mortgage_ai import config
from mortgage_ai.application import list_applications
from mortgage_ai.ingest import ingest_applications, ingest_regulations

ingest_regulations()
ingest_applications()
print("\nApplications:", list_applications())

## 5. Compliance check

Runs every rule, compares the typed data with the signed scan, and cites the policy section for each problem.

In [ ]:
from main import print_report
from mortgage_ai.compliance import check_application_folder
from mortgage_ai.storage import save_report

for app_id in list_applications():
    report = check_application_folder(app_id)
    save_report(report)
    print_report(report)

## 6. Ask questions

The first question loads the language model (~1 minute); after that answers are fast.
Questions about something that is not in the documents get *"Not found in the documents."*

In [ ]:
from main import print_answer
from mortgage_ai.qa import answer

for question in [
    "What is the borrower's base income?",
    "Who is the borrower's employer?",
    "Has the borrower declared bankruptcy in the last 7 years?",
    "What is the agency case number?",           # only on the scanned form
    "What is the borrower's favourite football team?",
]:
    print_answer(answer(question, application="APP-001"))

In [ ]:
# Ask your own question: edit and re-run this cell
QUESTION = "What is the loan amount?"
APPLICATION = "APP-001"          # or "APP-002"

print_answer(answer(QUESTION, application=APPLICATION))

In [ ]:
# Questions about the lending policy
print_answer(answer("What is the maximum debt-to-income ratio?", config.REGULATIONS_COLLECTION))
print_answer(answer("What is the maximum loan-to-value ratio for a second home?", config.REGULATIONS_COLLECTION))

## 7. Read a scanned form

Turns the scanned Form 1003 into structured data. Fields that cannot be read reliably
(unclear checkboxes) are left empty and listed, never guessed.

In [ ]:
import json
from mortgage_ai.application import load_scanned_form
from mortgage_ai.form_reader import unreadable_fields

scanned = load_scanned_form(config.APPLICATIONS_DIR / "APP-001" / "scanned_form.pdf")
print(json.dumps(scanned["loan_information"], indent=2))
print("\nCould not read:", unreadable_fields(scanned))

## 8. Check your own application

Set `UPLOAD = True`, run the cell and pick your files:
- `typed_form.json`: the form data (same fields as `data/applications/APP-002/typed_form.json`), and/or
- `scanned_form.pdf`: the signed Form 1003

In [ ]:
UPLOAD = False
NEW_APPLICATION_ID = "APP-100"

if UPLOAD and IN_COLAB:
    from google.colab import files
    folder = config.APPLICATIONS_DIR / NEW_APPLICATION_ID
    folder.mkdir(parents=True, exist_ok=True)
    for name, content in files.upload().items():
        (folder / os.path.basename(name)).write_bytes(content)
    ingest_applications()
    report = check_application_folder(NEW_APPLICATION_ID)
    save_report(report)
    print_report(report)

## 9. Accuracy

Asks 29 questions with known answers (`tests/eval_questions.json`).

In [ ]:
from mortgage_ai.evaluate import run_evaluation
accuracy = run_evaluation()

## 10. History and download

In [ ]:
from mortgage_ai.storage import recent_questions, recent_reports

print("Reports:")
for r in recent_reports(5):
    print(f"  {r['created_at']}  {r['application']:<8} {r['decision']}")
print("\nQuestions:")
for q in recent_questions(5):
    print(f"  [{q['application'] or 'policy'}] {q['question']}  ->  {q['answer']}")

In [ ]:
DOWNLOAD_REPORTS = True

if IN_COLAB and DOWNLOAD_REPORTS:
    import shutil
    from google.colab import files
    shutil.make_archive("/content/mortgage_reports", "zip", config.REPORTS_DIR)
    files.download("/content/mortgage_reports.zip")